## Case Demanding:

- Analysis Questions:
Provide answers to the following questions based on the transformed data:
Which color generated the highest revenue each year?
What is the average LeadTimeInBusinessDays by ProductCategoryName?

#### Environment Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import datetime

In [0]:
df_products = spark.table('interviewcaseupstart.sales_db.publish_product')
df_orders = spark.table('interviewcaseupstart.sales_db.publish_orders')

## Analysis Questions

###### Which color generated the highest revenue each year?

In [0]:
try:
    #Geting the year for each order
    df_color_year = df_orders.join(df_products,on='ProductID',how='inner')
    df_color_year = (df_color_year
                    .withColumn('OrderYear',F.year('OrderDate'))
                    .filter(F.col('OrderYear').isNotNull())
                    .select('Color','TotalLineExtendedPrice','OrderYear'))
    
    #Creating the row number for each color, based on year and sorting by revenue
    df_color_year = (df_color_year
                    .withColumn('row_number',
                                F.row_number().over(Window.partitionBy('Color','OrderYear').orderBy(F.desc('TotalLineExtendedPrice')))
                                )
                    .filter(F.col('row_number') == 1)
                    )
    
    #With the revenue for each year and color and filtering the top 1, we can get the color with the max revenue
    df_color_year = (df_color_year
                    .withColumn('row_number_revenue',
                                F.row_number().over(Window.partitionBy('OrderYear').orderBy(F.desc('TotalLineExtendedPrice'))))
                    .filter(F.col('row_number_revenue') == 1)
                    .select('Color','TotalLineExtendedPrice','OrderYear'))
    display(df_color_year)
except Exception as e:
  dbutils.notebook.exit(e)

###### What is the average LeadTimeInBusinessDays by ProductCategoryName?

In [0]:
try:
    #Filtering nulls
    df_avg_lead = df_orders.join(df_products,on='ProductID',how='inner')
    df_avg_lead = df_avg_lead.filter(F.col('LeadTimeInBusinessDays').isNotNull() & F.col('ProductCategoryName').isNotNull())
    
    #Getting the avg LeadTimeInBusinessDays for each category
    df_avg_lead = df_avg_lead.groupBy(F.col('ProductCategoryName')).avg('LeadTimeInBusinessDays').alias('AvgLeadTime')
    df_avg_lead = df_avg_lead.withColumn('AvgLeadTime', F.round(F.col('avg(LeadTimeInBusinessDays)'),2)).drop('avg(LeadTimeInBusinessDays)')
    display(df_avg_lead)
except Exception as e:
  dbutils.notebook.exit(e)